# Datakwaliteit Archeologische Complexenbestand
auteur: Thya van den Berg

Organisatie: Rijksdienst voor het cultureel erfgoed

Programma: DNA-NL

Project: Bronnen- en datakwaliteit

Projectleider: Maurice de Kleijn

In dit notebook wordt het bestand DEF_complexen_v8 beoordeeld op datakwaliteit, volgens de datakwaliteitsrichtlijnen van Bronnen- en Datakwaliteit. Dit staat naast de beoordeling en suggesties die gedaan zijn vanuit de afdeling Archeologie in het bestand "reparatie en homogenisering complexenbestand". DEF_complexen_v8 bevat archaeologische complexen welke geassocieerd zijn met archeologische rijksmonumenten. Het is een kennisbestand dat geen verdere wettelijke waarde heeft, maar het heeft wel identificerende waarde, omdat hierin de werkelijk aanwezige archeologie binnen- of direct geassocieerd aan een rijksmonument wordt omschreven.

## Installeer packages (niet relevant voor non-programmeerder)

In [184]:
# %pip install duckdb
# %pip install matplotlib
# %pip install mpl_toolkits
# %pip install shapely

## Laad packages (niet relevant voor non-programmeerder)

In [185]:
import duckdb
import inspect
import geopandas as gpd
import pandas as pd
import os
import matplotlib.pyplot as plt
from shapely import wkt


## Benodigde variabelen (niet relevant voor non-programmeerder)

### Voor medallion architecture

In [186]:
### Variabelen definiëren voor Medallion Architecture
datalake_path = os.path.normpath("./data/")
zilveren_laag_path = "zilver"

### Voor opslaan

In [187]:
# voor tussendoor opslaan
save_directory = os.path.normpath(r"./data/saved_parquet_files//")
if not os.path.exists(save_directory):
    # maak directory als deze nog niet bestaat
    os.mkdir(save_directory)

# voor opslaan "quick wins"
zilver_directory = os.path.normpath(f"./data/{zilveren_laag_path}//")
if not os.path.exists(zilver_directory):
    # maak directory als deze nog niet bestaat
    os.mkdir(zilver_directory)

## Benodigde functies (niet relevant voor non-programmeerder)

In [188]:
# sla output van een query op (template)
# file_name = 'query_output.parquet'
# duckdb.sql(f"""
#             COPY
#                 (SELECT x.*, y.*
#                 FROM
#                 brons_beschermde_stads_en_dorpsgezichten.townscapes x
#                 JOIN
#                 brons_rijksmonumentenregister.tblTEXT_OBJECT y
#                 ON
#                 x.bron_id = y.OBJ_NUMMER)
#                 TO '{os.path.join(save_directory, file_name)}'
#                 (FORMAT parquet);""")
#
# # alternatief
# duckdb.sql(f"""SELECT x.*, y.*
#                 FROM
#                 brons_beschermde_stads_en_dorpsgezichten.townscapes x
#                 JOIN
#                 brons_rijksmonumentenregister.tblTEXT_OBJECT y
#                 ON
#                 x.bron_id = y.OBJ_NUMMER""").write_parquet(os.path.join(save_directory, file_name))

def empty_duckdb_memory():
    """Empty the in-memory duckdb database"""
    schemas = duckdb.sql("""
                SELECT schema_name
                FROM information_schema.schemata
                WHERE schema_name NOT IN ('information_schema', 'pg_catalog', 'temp', 'main')""").fetchdf()
    for schema in schemas['schema_name']:
        tables = duckdb.sql(f"""
                        SELECT table_name
                        FROM information_schema.tables
                        WHERE table_schema = '{schema}'
        """).fetchdf()
        for table in tables['table_name']:
            duckdb.sql(f"DROP TABLE IF EXISTS {schema}.{table}")
        duckdb.sql(f"DROP SCHEMA IF EXISTS {schema}")


def save_complete_duckdb_memory(save_directory):
    """saves entire in-memory duckdb database to the given save_directory"""
    duckdb.sql(f"""EXPORT DATABASE '{save_directory}' (FORMAT parquet)""")

def load_duckdbset_to_memory(load_directory):
    """Loads an entire saved set of parquet files from the included "schema.sql" and "load.sql" files."""
    file_name = 'schema.sql'
    with open(os.path.join(load_directory, file_name), 'r') as schema:
        duckdb.sql(schema.read())

    file_name = 'load.sql'
    with open(os.path.join(load_directory, file_name), 'r') as load:
        duckdb.sql(load.read())
    duckdb.sql("SHOW SCHEMAS;")

## Duckdb setup (niet relevant voor non-programmeerder)
duckdb is een databasemanagementsoftware vergelijkbaar met PostGRES en MySQL. Het gebruikt SQL queries om met de data te interacteren. In dit notebook gaat dit volledig lokaal en in het werkgeheugen van de computer, anders dan in het DAP, echter, de queries die moeten worden uitgevoerd voor het verbeteren van de data zijn hetzelfde.

### Laad de Spatial Extension
https://duckdb.org/docs/current/core_extensions/spatial/overview

In [189]:
duckdb.sql("""
           INSTALL spatial;
           LOAD spatial;
           """)

### laad database (schema's) in de bronzen laag


In [190]:
duckdb.sql("""
           CREATE SCHEMA IF NOT EXISTS brons_archeologische_complexen;
           """)

### laad parquet files als een tabellen in de bronzen laag


In [191]:
duckdb.sql("""
           CREATE TABLE IF NOT EXISTS brons_archeologische_complexen.def_complexen_v8 AS
           FROM read_parquet('./data/brons/def_complexen_v8/def_complexen_v8.parquet');
           """)


## Maak test dataset

In [192]:
duckdb.sql("""DROP TABLE IF EXISTS test.test""")
duckdb.sql("""CREATE SCHEMA IF NOT EXISTS test;""")
duckdb.sql("""CREATE TABLE IF NOT EXISTS test.test (test_eenheid_grootheid VARCHAR);""")
duckdb.sql("""INSERT INTO test.test (test_eenheid_grootheid) VALUES ('1cm'), ('1 cm'), ('F23B'), ('23FB'), ('1cm1'), ('cm1');""")
df = duckdb.sql("""SELECT * FROM test.test;""").fetchdf()
df


,test_eenheid_grootheid
0,1cm
1,1 cm
2,F23B
3,23FB
4,1cm1
5,cm1


## Algemene omschrijvingen

Hier wordt de data in het algemeen uiteengezet. Dit zijn geen testen, maar er valt veel informatie terug te lezen over de opzet van de dataset.

### Omschrijf de eigenschappen van de kolommen.

In [193]:
df = duckdb.sql("""
        DESCRIBE brons_archeologische_complexen.def_complexen_v8;
        """).fetchdf()
df


,column_name,column_type,null,key,default,extra
0,complex_id,DOUBLE,YES,None,None,None
1,terreinnum,DOUBLE,YES,None,None,None
2,rijksmonum,DOUBLE,YES,None,None,None
3,cma,VARCHAR,YES,None,None,None
4,cma_volgnr,VARCHAR,YES,None,None,None
5,cpx_monito,BIGINT,YES,None,None,None
6,cpx_jaar,VARCHAR,YES,None,None,None
7,cpx_code_b,VARCHAR,YES,None,None,None
8,cpx_code_o,VARCHAR,YES,None,None,None
9,cpx_code_n,VARCHAR,YES,None,None,None


### Omschrijf de eigenschappen van de waarden in de kolommen.
Let op: veel van deze omschrijvingen zijn niet erg behulpzaam voor tekstkolommen. Dit zijn: min, max, avg, std, q25, q50 en q75.

In [194]:
duckdb.sql("""
        SUMMARIZE brons_archeologische_complexen.def_complexen_v8;""").fetchdf()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,complex_id,DOUBLE,20.0,50999.0,4482,14936.711100339311,20197.506587843258,1173.1558922558922,2332.3364877133054,16567.20975938507,4126,0.00
1,terreinnum,DOUBLE,42.0,20016.0,2300,2786.5399903053803,3964.6532419246237,671.9497071688043,1387.452600800246,1610.1471065440778,4126,0.00
2,rijksmonum,DOUBLE,31550.0,532469.0,1877,119046.30392632089,167622.54516092004,45475.557098765436,45788.64903107646,46174.0,4126,0.00
3,cma,VARCHAR,02G-026,geen CMA-nummer,1356,NaN,NaN,NaN,NaN,NaN,4126,6.62
4,cma_volgnr,VARCHAR,12A-001-01,?,1645,NaN,NaN,NaN,NaN,NaN,4126,55.84
5,cpx_monito,BIGINT,0,5220,3415,1746.8800290838585,1241.7678228908678,718,1538,2801,4126,0.00
6,cpx_jaar,VARCHAR,20150101000000.000,20170912000000.000,4,NaN,NaN,NaN,NaN,NaN,4126,92.46
7,cpx_code_b,VARCHAR,(Ring)walburg,weg,62,NaN,NaN,NaN,NaN,NaN,4126,0.00
8,cpx_code_o,VARCHAR,EGMW,VX,64,NaN,NaN,NaN,NaN,NaN,4126,0.51
9,cpx_code_n,VARCHAR,APVV,nederzettingsresten Swifterbantcultuur,76,NaN,NaN,NaN,NaN,NaN,4126,0.48


# Pas de datakwaliteits richtlijnen toe



In [195]:
# algemene variabelen
table =  "brons_archeologische_complexen.def_complexen_v8"
schema = "brons_archeologische_complexen"
table_name = 'def_complexen_v8'
dk_dimensies_voldaan = 0

## Dimensie 1: referentiële integriteit en unieke identificatie


- Vraag: Welke kolom bevat de primary key/ID?
    - Test A: Alle IDs in de primary key kolom zijn uniek.
    - Test B: Er zijn geen lege waarden in de primary key kolom.
    - Test C: In de databasestructuur is aangegeven dat de primary key kolom uniek moet zijn.
    - Test D: In de databasestructuur is aangegeven dat de primary key geen lege waarden mag bevatten.
    - Test E: In de databasestructuur is aangegeven dat de kolom een primary key is.
- Vraag: Welke kolom(men) bevat(ten) foreign key(s)?
    - Vraag: Naar welke dataset verwijst/verwijzen deze foreign key(s)?
        - Test F: Het is bekend van elke foreign key waar deze naar verwijst
    - Vraag: Is er een losse kolom voor elke foreign key? (note Thya: dit zou ik graag automatiseren en in een test veranderen, maar ik zie nog niet goed hoe.)
        - Test G: De foreign key(s) kom(t/en) overeen met een primary key uit de dataset waar deze naar verwijst.
    - Hebben alle foreign keys een eigen kolom?

#### Vraag: Welke kolom bevat de primary key?

In [196]:
primary_id_kolom = "complex_id"
test_resultaten_dim1 = []
print(f"de volgende kolom bevat de primary key: {primary_id_kolom}")

de volgende kolom bevat de primary key: complex_id


#### Test A

In [197]:
pk_is_unique = duckdb.sql(f"""
SELECT(
    SELECT
        COUNT()
    FROM (
        SELECT
            DISTINCT {primary_id_kolom}
        FROM {table}
    )
) == (
    SELECT
        COUNT()
    FROM {table}
)
AS unique_identifier;
""").fetchone()[0]
print(f"Test A :Alle IDs in de primary ID kolom zijn uniek: {pk_is_unique}")
test_resultaten_dim1.append(pk_is_unique)


Test A :Alle IDs in de primary ID kolom zijn uniek: True


#### Test B

In [198]:
pk_is_not_null = duckdb.sql(f"""
SELECT (
    SELECT
        COUNT()
    FROM {table}
    WHERE {primary_id_kolom} IS NULL
    ) = 0
AS null_identifier;
""").fetchone()[0]
print(f"Test B: Er zijn geen lege waarden in de primary id kolom: {pk_is_not_null}")
test_resultaten_dim1.append(pk_is_not_null)

Test B: Er zijn geen lege waarden in de primary id kolom: True


#### Test C

In [199]:
pk_schema_unique = duckdb.sql(f"""
SELECT
    CASE WHEN EXISTS(
        SELECT
            tc.constraint_type
        FROM
            information_schema.table_constraints tc
        JOIN
            information_schema.key_column_usage kcu
            ON tc.constraint_name = kcu.constraint_name
            AND tc.table_schema = kcu.table_schema
            AND tc.table_name = kcu.table_name
        WHERE
            tc.table_schema = '{schema}'
            AND tc.table_name = '{table_name}'
            AND kcu.column_name = '{primary_id_kolom}'
            AND tc.constraint_type = 'UNIQUE')
    THEN True
    ELSE False
    END
AS has_unique_constraint;
""").fetchone()[0]
print(f"Test C: In de databasestructuur is aangegeven dat de primary key kolom unique moet zijn: {pk_schema_unique}")
test_resultaten_dim1.append(pk_schema_unique)

Test C: In de databasestructuur is aangegeven dat de primary key kolom unique moet zijn: False


#### Test D

In [200]:
pk_schema_not_null = duckdb.sql(f"""
SELECT(
    SELECT is_nullable FROM information_schema.columns
        WHERE table_schema = '{schema}' AND table_name = '{table_name}' AND column_name = '{primary_id_kolom}'
    ) = 'NO' as not_nullable;""").fetchone()[0]
print(f"Test D: In de databasestructuur is aangegeven dat de primary key kolom niet null mag zijn: {pk_schema_not_null}")
test_resultaten_dim1.append(pk_schema_not_null)



Test D: In de databasestructuur is aangegeven dat de primary key kolom niet null mag zijn: False


#### Test E

In [201]:
pk_schema_exists = duckdb.sql(f"""
SELECT
    CASE WHEN EXISTS (
        SELECT
            1
        FROM
            information_schema.table_constraints tc
        JOIN
            information_schema.key_column_usage kcu
            ON tc.constraint_name = kcu.constraint_name
            AND tc.table_schema = kcu.table_schema
            AND tc.table_name = kcu.table_name
        WHERE
            tc.table_schema = 'schema1'
            AND tc.table_name = 'table1'
            AND kcu.column_name = 'column1'
            AND tc.constraint_type = 'PRIMARY KEY'
    ) THEN TRUE
    ELSE FALSE
    END
AS is_primary_key;
""").fetchone()[0]
print(f"Test E: In de databasestructuur is aangegeven dat de primary key een primary key is: {pk_schema_exists}")
test_resultaten_dim1.append(pk_schema_exists)

Test E: In de databasestructuur is aangegeven dat de primary key een primary key is: False


#### Vragen: Welke kolom(men) bevat(ten) foreign key(s) ; Naar welke dataset verwijst/verwijzen deze foreign key(s)?
handmatig ingevoerd op basis van domeinkennis, dit is niet met code te achterhalen.
Let op: nog geen code geschreven om ook een kolom waar meerdere gemixte foreign keys in staan te checken, omdat dit voor deze dataset niet nodig is. Komt later. #TODO

In [202]:
# in de eerste waarde ([0]) van de lijst staat naar welke dataset/database verwezen wordt, in de tweede [1] staat het schema van hoe deze op dit moment is ingeladen in brons, in de derde [2] de tabelnaam in brons, en de vierde [3] de kolomnaam van de primary key van die dataset en de vijfde [4] de opgeslagen parquet file óf de waarde None als het niet om een vindbare dataset gaat.
dict_foreign_keys = {
    "terreinnum": ["DEF_terreinen_v5", "brons_archeologische_terreinen", "def_terreinen_v5",  "terreinnum", r"./data/brons/def_terreinen_v5/def_terreinen_v5.parquet"],
    "rijksmonum": ["rijksmonumentenregister", "brons_rijksmonumentenregister", "tblTEXT_OBJECT", "TXO_TEXT_KEY", r"./data/brons/rijksmonumentenregister/tblTEXT_OBJECT.parquet"],
    "cma":["Centraal_monumenten_archief", None, None, None, None], #zit in Proza, niet te testen. Dit is een gedigitaliseerd/gscand papieren archief. Geen verantwoordelijke, maar vraag Guide Mauro
    "cma_volgnr": ["Centraal_monumenten_archief", None, None, None, None], # zit in Proza, niet te testen
    "cpx_monito": [None,None, None, None, None] # Herkomst onbekend. In (verhouderde) data dictionary staat Unieke identifier RCE monitor, maar welke? Niet de BAAC nulmonitor en niet de huidige monitor.
}
for item in dict_foreign_keys:
    if dict_foreign_keys[item][0]:
        if dict_foreign_keys[item][4]:
            print(f"kolom {item} verwijst naar de primary key van {dict_foreign_keys[item][0]}, deze staat in kolom {dict_foreign_keys[item][3]} van {dict_foreign_keys[item][1]}.{dict_foreign_keys[item][2]}")
        else:
            print(f"kolom {item} verwijst naar een key van {dict_foreign_keys[item][0]}, deze dataset is niet ingeladen in de omgeving.")
    else:
        print(f"kolom {item} verwijst naar een onbekende dataset.")


kolom terreinnum verwijst naar de primary key van DEF_terreinen_v5, deze staat in kolom terreinnum van brons_archeologische_terreinen.def_terreinen_v5
kolom rijksmonum verwijst naar de primary key van rijksmonumentenregister, deze staat in kolom TXO_TEXT_KEY van brons_rijksmonumentenregister.tblTEXT_OBJECT
kolom cma verwijst naar een key van Centraal_monumenten_archief, deze dataset is niet ingeladen in de omgeving.
kolom cma_volgnr verwijst naar een key van Centraal_monumenten_archief, deze dataset is niet ingeladen in de omgeving.
kolom cpx_monito verwijst naar een onbekende dataset.


#### Test F

In [203]:
fk_ref_known = True

for item in dict_foreign_keys:
    if not dict_foreign_keys[item][0]:
        fk_ref_known = False
print(f"Test F: Het is voor elke foreign key bekend welke dataset deze naar refereert: {fk_ref_known}")
test_resultaten_dim1.append(fk_ref_known)

Test F: Het is voor elke foreign key bekend welke dataset deze naar refereert: False


#### Test G

laad de benodigde datasets in zodat er een vergelijking gemaakt kan worden

In [204]:
for item in dict_foreign_keys:
    if dict_foreign_keys[item][4]:
        duckdb.sql(f"""
                   CREATE SCHEMA IF NOT EXISTS {dict_foreign_keys[item][1]};
                   """)
        duckdb.sql(f"""
                   CREATE TABLE IF NOT EXISTS {dict_foreign_keys[item][1]}.{dict_foreign_keys[item][2]} AS
                   FROM read_parquet('{dict_foreign_keys[item][4]}');
                   """)
    else:
        print(f"LET OP!: dataset {dict_foreign_keys[item][0]} voor foreign key {item} wordt niet gecheckt.")


LET OP!: dataset Centraal_monumenten_archief voor foreign key cma wordt niet gecheckt.
LET OP!: dataset Centraal_monumenten_archief voor foreign key cma_volgnr wordt niet gecheckt.
LET OP!: dataset None voor foreign key cpx_monito wordt niet gecheckt.


In [205]:
# test hier of het inladen gelukt is:
# duckdb.sql(f"""DESCRIBE brons_archeologische_terreinen.def_terreinen_v5;""")
# duckdb.sql(f"""DESCRIBE brons_rijksmonumentenregister.tblTEXT_OBJECT;""")

In [206]:
fk_all_match = True
for item in dict_foreign_keys:
    if dict_foreign_keys[item][4]:
        val = duckdb.sql(f"""
        SELECT
            LIST_HAS_ALL((
                SELECT
                    LIST({dict_foreign_keys[item][3]})
                FROM {dict_foreign_keys[item][1]}.{dict_foreign_keys[item][2]}),
                (SELECT
                    LIST({item})
                FROM {table})
        );
        """).fetchone()[0]
        print(f"alle waarden in de foreign key kolom {item} komen voor in de kolom {dict_foreign_keys[item][3]} van {dict_foreign_keys[item][1]}.{dict_foreign_keys[item][2]}: {val}")
        if not val:
            fk_all_match = False
print(f"De foreign key(s) kom(t/en) overeen met een primary key(s) uit de dataset waar deze naar verwijst/verwijzen: {fk_all_match}")
test_resultaten_dim1.append(fk_all_match)


alle waarden in de foreign key kolom terreinnum komen voor in de kolom terreinnum van brons_archeologische_terreinen.def_terreinen_v5: True
alle waarden in de foreign key kolom rijksmonum komen voor in de kolom TXO_TEXT_KEY van brons_rijksmonumentenregister.tblTEXT_OBJECT: False
De foreign key(s) kom(t/en) overeen met een primary key(s) uit de dataset waar deze naar verwijst/verwijzen: False


#### Resultaten dimensie 1

In [207]:
aantal_testen_succes_dim1 = 0

for test in test_resultaten_dim1:
    if test:
        aantal_testen_succes_dim1 += 1

if len(test_resultaten_dim1) == aantal_testen_succes_dim1:
    print(f"Er zijn {len(test_resultaten_dim1)} tests uitgevoerd voor dimensie 1. Daarvan is/zijn aan {aantal_testen_succes_dim1} test(en) voldaan. Dat betekent dat aan datakwaliteitsdimensie 1 in zijn geheel is voldaan")
    dk_dimensies_voldaan +=1
else:
    print(f"Er zijn {len(test_resultaten_dim1)} tests uitgevoerd voor dimensie 1. Daarvan is/zijn aan {aantal_testen_succes_dim1} test(en) voldaan. Dat betekent dat aan datakwaliteitsdimensie 1 NIET in zijn geheel is voldaan.")



Er zijn 7 tests uitgevoerd voor dimensie 1. Daarvan is/zijn aan 2 test(en) voldaan. Dat betekent dat aan datakwaliteitsdimensie 1 NIET in zijn geheel is voldaan.


## Dimensie 2: Structurele consistentie
* Vraag: Is het datatype voor elke kolom logisch voor de informatie die in die kolom staat?
    * Test A: Er zijn geen kolommen die volledig uit data van een ander type dan het aangegeven type bestaan.
        * A.1 floats die integers zouden kunnen zijn
        * A.2 integers die booleans zouden kunnen zijn
        * A.3 tekst die numeric (float of integer) zouden kunnen zijn?
* Vraag: Bevat elke kolom maar 1 datatype?
    * Test B: Er zijn geen tekstkolommen die data van een ander type bevatten
* Vraag: Bevat elke kolom maar 1 stuk informatie?
    * Test C: Er zijn geen tekstkolommen die bestaan uit een combinatie van grootheden met eenheden.
    * Test D: Alle kolommen hebben een andere naam.
    * Test E: Er zijn geen kolommen met volledig gedupliceerde informatie.
* Vraag: Is het gebruik van onbekende waarden consistent?
    * Test F: Er is maximaal 1 waarde voor informatie die niet is ingevuld (1 van: NULL, NA, n/a, None, "")
    * Test G: Er is maximaal 1 waarde per datatype voor informatie die onbekend is. (bijvoorbeeld ?, -, "onbekend", 9999, 0000, 01-01-1901)

In [208]:
# get all the datatypes to be tested
list_datatypes =[name for name, value in inspect.getmembers(duckdb.sqltypes)
    if not name.startswith("_")
]
print("de volgende datatypen zijn beschikbaar in duckdb")
list_datatypes

de volgende datatypen zijn beschikbaar in duckdb


['BIGINT',
 'BIT',
 'BLOB',
 'BOOLEAN',
 'DATE',
 'DOUBLE',
 'DuckDBPyType',
 'FLOAT',
 'HUGEINT',
 'INTEGER',
 'INTERVAL',
 'SMALLINT',
 'SQLNULL',
 'TIME',
 'TIMESTAMP',
 'TIMESTAMP_MS',
 'TIMESTAMP_NS',
 'TIMESTAMP_S',
 'TIMESTAMP_TZ',
 'TIME_NS',
 'TIME_TZ',
 'TINYINT',
 'UBIGINT',
 'UHUGEINT',
 'UINTEGER',
 'USMALLINT',
 'UTINYINT',
 'UUID',
 'VARCHAR',
 'VARIANT']

In [209]:

# remove unnecesary datatypes
print("De volgende datatypen worden gegroepeerd:")

# group datatypes that are tested in the same way
dict_datatypes = {
    'integer': ['BIGINT', 'HUGEINT','INTEGER','SMALLINT','TINYINT','UBIGINT','UHUGEINT','UINTEGER','USMALLINT','UTINYINT','UUID'],
    'float': ['DOUBLE','FLOAT'],
    'decimal': ['DECIMAL'],
    'datetime': ['DATE', 'TIMESTAMP', 'TIME', 'TIMESTAMP', 'TIMESTAMP_MS', 'TIMESTAMP_NS', 'TIMESTAMP_S', 'TIMESTAMP_TZ', 'TIME_NS', 'TIME_TZ'],
    'boolean': ['BOOLEAN'],
    'string': ['VARCHAR']

}
dict_datatypes

De volgende datatypen worden gegroepeerd:


{'integer': ['BIGINT',
  'HUGEINT',
  'INTEGER',
  'SMALLINT',
  'TINYINT',
  'UBIGINT',
  'UHUGEINT',
  'UINTEGER',
  'USMALLINT',
  'UTINYINT',
  'UUID'],
 'float': ['DOUBLE', 'FLOAT'],
 'decimal': ['DECIMAL'],
 'datetime': ['DATE',
  'TIMESTAMP',
  'TIME',
  'TIMESTAMP',
  'TIMESTAMP_MS',
  'TIMESTAMP_NS',
  'TIMESTAMP_S',
  'TIMESTAMP_TZ',
  'TIME_NS',
  'TIME_TZ'],
 'boolean': ['BOOLEAN'],
 'string': ['VARCHAR']}

#### Test A

In [210]:
datatype_all_match = True
for item in dict_datatypes:
    if item == 'integer':
        # check if integer may be a boolean
        for type in dict_datatypes[item]:
            list_of_cols = duckdb.sql(f"""
                                            SELECT column_name FROM (DESCRIBE {table}) WHERE column_type ==  '{type}'
                                            """).fetchall()
            for item in list_of_cols:
                integer_contains_bool = duckdb.sql(f"""
                SELECT
                    CASE WHEN
                        (SELECT
                            COUNT({item[0]})
                        FROM
                            {table}
                        WHERE {item[0]} == 0
                        OR {item[0]} == 1
                        )
                        ==
                        (SELECT COUNT({item[0]}) FROM {table})
                        THEN True
                        ELSE False
                        END;
                """).fetchone()[0]
                if integer_contains_bool:
                    print(f'column {item[0]} with datatype {type} contains only 1 and 0 and is likely a boolean. The datatype is likely wrong')
                    datatype_all_match = False
                else:
                    print(f'column {item[0]} with datatype {type} is likely of the correct type.')


    elif item == 'float':
        # check if floating numbers may actually be integers. Are there any non-0 decimal numbers?
        # if more than 50% of the numbers are like this, the data type may be mixed. #TODO is this the correct cut-off?
        for type in dict_datatypes[item]:
            # query checks if column with type exists and returns columns
            list_of_cols = duckdb.sql(f"""
                                            SELECT column_name FROM (DESCRIBE {table}) WHERE column_type ==  '{type}'
                                            """).fetchall()
            for item in list_of_cols:

                float_contains_integer = duckdb.sql(f"""
                SELECT
                    CASE WHEN (
                    (SELECT
                        COUNT({item[0]})
                    FROM {table}
                    WHERE ({item[0]} % 1) == 0
                    )
                    >
                    (SELECT 0.5* COUNT() FROM {table}))
                    THEN True
                    ELSE False
                    END;
                """).fetchone()[0]

                float_contains_only_int = duckdb.sql(f"""
                SELECT
                    CASE WHEN (
                    (SELECT
                        COUNT({item[0]})
                    FROM {table}
                    WHERE ({item[0]} % 1) == 0
                    )
                    ==
                    (SELECT COUNT() FROM {table}))
                    THEN True
                    ELSE False
                    END;
                """).fetchone()[0]
                if float_contains_integer:
                    if float_contains_only_int:
                        print(f'column {item[0]} with datatype {type} contains no values with decimal points. The datatype is likely wrong')
                        datatype_all_match = False
                    else:
                        print(f'WARNING: column {item[0]} with type {type} contains more than 50% values that could be INTEGERS, the datatype may be incorrect but the test is not considered failed')
                else:
                    print(f'column {item[0]} with type {type} contains less than 50% values that could be INTEGERS, the data type is likely correct')
    elif item == 'string':
        for type in dict_datatypes[item]:
            list_of_cols = duckdb.sql(f"""
                                            SELECT column_name FROM (DESCRIBE {table}) WHERE column_type ==  '{type}'
                                            """).fetchall()
            for item in list_of_cols:
                # check if some strings contain only numbers
                string_contains_only_numeric = duckdb.sql(f"""
                SELECT
                    CASE WHEN (
                        SELECT
                            COUNT({item[0]})
                        FROM
                            {table}
                        WHERE
                            regexp_full_match({item[0]}, '[^0-9]')
                    )
                    >
                    (SELECT
                        0.5 * COUNT()
                    FROM
                        {table})
                    THEN True
                    ELSE False
                    END;
                """).fetchone()[0]

                if string_contains_only_numeric:
                        print(f'column {item[0]} with datatype {type} contains only values that consists of numbers. The datatype is likely wrong')
                        datatype_all_match = False
                else:
                    print(f'column {item[0]} with datatype {type} is likely of the correct type')
    else:
        print(f"datatype {item} is untested")

print(f"Test A: Er zijn geen kolommen die volledig uit data van een ander type dan het aangegeven type bestaan: {fk_ref_known}")



column cpx_monito with datatype BIGINT is likely of the correct type.
column complex_id with datatype DOUBLE contains no values with decimal points. The datatype is likely wrong
column terreinnum with datatype DOUBLE contains no values with decimal points. The datatype is likely wrong
column rijksmonum with datatype DOUBLE contains no values with decimal points. The datatype is likely wrong
column cpx_diepte with datatype DOUBLE contains no values with decimal points. The datatype is likely wrong
column cpx_x_coor with type DOUBLE contains less than 50% values that could be INTEGERS, the data type is likely correct
column cpx_y_coor with type DOUBLE contains less than 50% values that could be INTEGERS, the data type is likely correct
datatype decimal is untested
datatype datetime is untested
datatype boolean is untested
column cma with datatype VARCHAR is likely of the correct type
column cma_volgnr with datatype VARCHAR is likely of the correct type
column cpx_jaar with datatype VARCH

#### Test B

In [211]:
# komen er waarden van andere typen voor in de kolom
string_contains_only_string = True
for item in dict_datatypes:
    if item == 'string':
        for type in dict_datatypes[item]:
            list_of_cols = duckdb.sql(f"""
                                            SELECT column_name FROM (DESCRIBE {table}) WHERE column_type == '{type}'
                                            """).fetchall()
            for item in list_of_cols:
            # check voor integers
                string_contains_numeric = duckdb.sql(f"""
                SELECT
                    CASE WHEN EXISTS(
                        SELECT
                            {item[0]}
                        FROM
                            {table}
                        WHERE
                            regexp_full_match({item[0]}, '[0-9]+')
                    )
                    THEN True
                    ELSE False
                    END;
                """).fetchone()[0] # TODO my regex should probably be checked by someone who knows what they're doing...
            # check for float
                string_contains_float = duckdb.sql(fr"""
                SELECT
                    CASE WHEN EXISTS(
                        SELECT
                            {item[0]}
                        FROM
                            {table}
                        WHERE
                            regexp_full_match({item[0]}, '[0-9\.,]+') AND regexp_matches({item[0]}, '[,\.]+')
                    )
                    THEN True
                    ELSE False
                    END;
                """).fetchone()[0] # TODO my regex should probably be checked by someone who knows what they're doing...
            # check for scientific notation numbering
                string_contains_scinum = duckdb.sql(fr"""
                SELECT
                    CASE WHEN EXISTS(
                        SELECT
                            {item[0]}
                        FROM
                            {table}
                        WHERE
                            regexp_full_match({item[0]}, '[0-9E\.,-]+', 'i') AND regexp_matches({item[0]}, 'E', 'i')
                    )
                    THEN True
                    ELSE False
                    END;
                """).fetchone()[0] # TODO my regex should probably be checked by someone who knows what they're doing...
            # check voor boolean
                string_contains_bool = duckdb.sql(f"""
                SELECT
                    CASE WHEN EXISTS(
                        SELECT
                            1
                        FROM
                            {table}
                        WHERE
                            regexp_full_match({item[0]}, '0|1|True|False|Ja|Nee|y|n|j', 'i')
                    )
                    THEN True
                    ELSE False
                    END;
                """).fetchone()[0] # TODO my regex should probably be checked by someone who knows what they're doing...
            # check for date
                regex = r'^\d{4}-\d{2}-\d{2}$|^(0[1-9]|1[0-2])\/(0[1-9]|[12]\d|3[01])\/(19|20)\d{2}$|^(0[1-9]|[12]\d|3[01])\/(0[1-9]|1[0-2])\/(19|20)\d{2}$|^(0[1-9]|[12]\d|3[01])\.(0[1-9]|1[0-2])\.(19|20)\d{2}$' # TODO my regex should probably be checked by someone who knows what they're doing...
                string_contains_date = duckdb.sql(f"""
                SELECT
                    CASE WHEN EXISTS(
                        SELECT
                            1
                        FROM
                            {table}
                        WHERE
                            regexp_full_match({item[0]}, '{regex}', 'i')
                    )
                    THEN True
                    ELSE False
                    END;
                """).fetchone()[0]
                if not string_contains_date and not string_contains_bool and not string_contains_scinum and not string_contains_float and not string_contains_numeric:
                    print(f'kolom {item[0]} met het type {type} bevat geen waarden van het verkeerde type')
                else:
                    string_contains_only_string = False

                    concat_string = f"kolom {item[0]} met het type {type} heeft mogelijk waarden van het verkeerde type, namelijk:"
                    if string_contains_date:
                        concat_string += ' \ndatum '
                    if string_contains_bool:
                        concat_string += ' \nboolean '
                    if string_contains_scinum:
                        concat_string += ' \ncijfers in wetenschappelijke notering '
                    if string_contains_float:
                        concat_string += ' \nfloat '
                    if string_contains_numeric:
                        concat_string += ' \ncijfer '
                    print( concat_string)

print(f"Test B: Er zijn geen tekstkolommen die data van een ander type bevatten: {string_contains_only_string}. Let op, mogelijk zijn de waarden niet incorrect maar lijken ze op een ander type waarde")

kolom cma met het type VARCHAR heeft mogelijk waarden van het verkeerde type, namelijk: 
cijfers in wetenschappelijke notering  
cijfer 
kolom cma_volgnr met het type VARCHAR heeft mogelijk waarden van het verkeerde type, namelijk: 
cijfers in wetenschappelijke notering 
kolom cpx_jaar met het type VARCHAR heeft mogelijk waarden van het verkeerde type, namelijk: 
float 
kolom cpx_code_b met het type VARCHAR bevat geen waarden van het verkeerde type
kolom cpx_code_o met het type VARCHAR bevat geen waarden van het verkeerde type
kolom cpx_code_n met het type VARCHAR bevat geen waarden van het verkeerde type
kolom cpx_begin met het type VARCHAR bevat geen waarden van het verkeerde type
kolom cpx_eind_p met het type VARCHAR bevat geen waarden van het verkeerde type
kolom cpx_zichtb met het type VARCHAR heeft mogelijk waarden van het verkeerde type, namelijk: 
boolean 
kolom cpx_beschr met het type VARCHAR bevat geen waarden van het verkeerde type
kolom cpx_kwets met het type VARCHAR heeft 

#### Test C

In [212]:
# zijn er kolommen die grootheden en eenheden in dezelfde kolom hebben staan
print("de volgende eenheden worden getest. Er moet voldaan worden aan een patroon [getal] [optionele spatie] [eenheid]")
SI_base_units = ["s", "m", "g", "A", "K" ,"mol", "cd"] # kg vervangen met g voor flexibiliteit
list_prefixes = ["k" ,"h" ,"da" ,"d" ,"c" ,"m" ,"µ"]
list_additional_units = ["%", "°C", "h", "u" ]
units = SI_base_units+list_additional_units

for prefix in list_prefixes:
    for unit in ["m", "g"]:
        units.append(prefix+unit)

print(units) # TODO Thya: mogelijk moet deze op basis van wat we daadwerkelijke tegenkomen nog worden verbeterd. Ik heb in ieder geval niet geprobeerd om compleet te zijn.

de volgende eenheden worden getest. Er moet voldaan worden aan een patroon [getal] [optionele spatie] [eenheid]
['s', 'm', 'g', 'A', 'K', 'mol', 'cd', '%', '°C', 'h', 'u', 'km', 'kg', 'hm', 'hg', 'dam', 'dag', 'dm', 'dg', 'cm', 'cg', 'mm', 'mg', 'µm', 'µg']


In [213]:
str_no_nr_with_unit = True
for item in dict_datatypes:
    if item == 'string':
        for type in dict_datatypes[item]:
            list_of_cols = duckdb.sql(f"""
                                            SELECT column_name FROM (DESCRIBE {table}) WHERE column_type == '{type}'
                                            """).fetchall()
            for item in list_of_cols:
                # check eerst of er gesplitst kan worden in nummer [optionele spatie] niet nummer
                # check dan of de tekst match met een van de units

                string_contains_nr_with_unit = duckdb.sql(rf"""
                SELECT
                    CASE WHEN EXISTS
                    (
                        SELECT
                            1
                        FROM
                            {table}
                        WHERE
                            (
                                (
                                regexp_matches({item[0]}, '\s([a-zA-Z]*)$')
                                AND
                                regexp_matches({item[0]}, '^([\d,.])')
                                )
                                OR
                                (
                                regexp_matches({item[0]}, '^([\d,.])')
                                AND
                                regexp_matches({item[0]}, '([a-zA-Z])$')
                                )
                            )
                            AND
                                regexp_extract({item[0]}, '([a-zA-Z])$') IN {units}
                    )
                    THEN True
                    ELSE False
                    END;
                """).fetchone()[0]
                if string_contains_nr_with_unit:
                    print(f"{item[0]} met datatype {type} bevat mogelijk een combinatie van eenheden en grootheden")
                    str_no_nr_with_unit = False
print(f"Test C: Er zijn geen tekstkolommen die mogelijk bestaan uit een combinatie van grootheden met eenheden: {str_no_nr_with_unit}. Een verdere check is nodig om dit te controleren.")


cma met datatype VARCHAR bevat mogelijk een combinatie van eenheden en grootheden
Test C: Er zijn geen tekstkolommen die mogelijk bestaan uit een combinatie van grootheden met eenheden: False. Een verdere check is nodig om dit te controleren.


In [214]:
# test hier de code
#
# string_contains_bool = duckdb.sql(rf"""
#                         SELECT
#                         test_eenheid_grootheid,
#                         regexp_extract(test_eenheid_grootheid, '([a-zA-Z]*)$')
#                         FROM
#                             test.test
#                         WHERE
#                             (
#                                 (regexp_matches(test_eenheid_grootheid, '\s([a-zA-Z]*)$') AND regexp_matches(test_eenheid_grootheid, '^([\d,.])'))
#                                 OR
#                                 (
#                                 regexp_matches(test_eenheid_grootheid, '^([\d,.])')
#                                 AND
#                                 regexp_matches(test_eenheid_grootheid, '([a-zA-Z])$')
#                                 )
#                             )
#                             AND
#                                 regexp_extract(test_eenheid_grootheid, '([a-zA-Z])$') IN {units};
#                 """).fetchdf()
# string_contains_bool

#### Test D

In [215]:
# duckdb hernoemt kolommen met dezelfde naam door er _1, _2 etc aan toe te voegen.
duplicate_colnames = duckdb.sql(rf"""
SELECT
    CASE WHEN EXISTS(
        SELECT
            1
        FROM
            (DESCRIBE {table})
        WHERE
            regexp_matches(column_name, '_\d$')
            )
    THEN False
    ELSE True
    END;
    """).fetchone()[0]
print(f"Test D: Alle kolommen hebben een andere naam: {duplicate_colnames}")

Test D: Alle kolommen hebben een andere naam: True


#### Test E

In [216]:
duplicate_column_info = False
# TODO deze heb ik even overgeslagen omdat het een hele omslachtige query werd, terwijl het niet zo'n hele belangrijke test is.
print(f"Test E: Er zijn geen kolommen met volledig gedupliceerde informatie: {duplicate_column_info}")

Test E: Er zijn geen kolommen met volledig gedupliceerde informatie: False


#### Test F

In [217]:
list_text_leeg = ['NULL', "None", "-", "/", "Geen", "N/A", "", "niks", "undefined", "empty", "leeg" ] # TODO hier vallen vast nog opties aan toe te voegen.
no_multiple_empty_vals = True
number_of_vals_present = 0
for item in dict_datatypes:
    if item == 'string':
        for type in dict_datatypes[item]:
            list_of_cols = duckdb.sql(f"""
                                            SELECT column_name FROM (DESCRIBE {table}) WHERE column_type == '{type}'
                                            """).fetchall()
            for item in list_of_cols:
                for empty_val in list_text_leeg:
                    val_present = duckdb.sql(f"""
                    SELECT
                        CASE WHEN EXISTS(
                            SELECT 1
                            FROM {table}
                            WHERE {item[0]} == '{empty_val}')
                        THEN True
                        ELSE False
                        END;
                    """)
                    if val_present:
                        break
                if val_present:
                    number_of_vals_present +=1
                    print(f"Waarde {empty_val} is in gebruik in een tekstveld")
                    break # break uit de for loop zodat er niet meer dan 1x per mogelijkheid wordt opgeteld.
if number_of_vals_present > 1:
    no_multiple_empty_vals = False

print(f"Test F: Er is maximaal 1 waarde voor informatie die niet is ingevuld: {no_multiple_empty_vals}")

Waarde NULL is in gebruik in een tekstveld
Test F: Er is maximaal 1 waarde voor informatie die niet is ingevuld: True


#### Test G

In [218]:
list_text_unknowns = ["?", "onbekend", "unknown", "??", "???", "????", "weet niet"] # TODO hier vallen vast nog opties aan toe te voegen.
list_datetime_unknowns = ["9999-99-99", "1901-01-01", "0000-00-00 00:00:00", "0000-00-00", "9999-99-99 99:99:99"] # TODO hier vallen vast nog opties aan toe te voegen.
# TODO nog geen lijst voor integers en floats. Ik weet dat hier ook bijvoorbeeld 99999 en dergelijken in gebruik zouden kunnen zijn, maar ik kon niet bedenken hoeveel en wanneer dat nou wel of niet de test zou moeten triggeren.
no_multiple_unknown_vals = False
number_of_vals_present_text = 0
number_of_vals_present_datetime = 0
for item in dict_datatypes:
    if item == 'string':
        for type in dict_datatypes[item]:
            list_of_cols = duckdb.sql(f"""
                                            SELECT column_name FROM (DESCRIBE {table}) WHERE column_type == '{type}'
                                            """).fetchall()
            for item in list_of_cols:
                for unknown_val in list_text_unknowns:
                    val_present = duckdb.sql(f"""
                    SELECT
                        CASE WHEN EXISTS(
                            SELECT 1
                            FROM {table}
                            WHERE {item[0]} == '{unknown_val}')
                        THEN True
                        ELSE False
                        END;
                    """)
                    if val_present:
                        break
                if val_present:
                    number_of_vals_present_text +=1
                    print(f"Waarde {unknown_val} is in gebruik in een tekstveld")
                    break # break uit de for loop zodat er niet meer dan 1x per mogelijkheid wordt opgeteld.
        if number_of_vals_present_text > 1:
            no_multiple_empty_vals = False
            break
    elif item == 'datetime':
        for type in dict_datatypes[item]:
            list_of_cols = duckdb.sql(f"""
                                        SELECT column_name FROM (DESCRIBE {table}) WHERE column_type == '{type}'
                                        """).fetchall()
            for item in list_of_cols:
                for unknown_val in list_datetime_unknowns:
                    val_present = duckdb.sql(f"""
                        SELECT
                            CASE WHEN EXISTS(
                                SELECT 1
                                FROM {table}
                                WHERE {item[0]} == '{unknown_val}')
                            THEN True
                            ELSE False
                            END;
                        """)
                    if val_present:
                            break
                if val_present:
                    number_of_vals_present_datetime +=1
                    print(f"Waarde {empty_val} is in gebruik in een tekstveld")
                    break # break uit de for loop zodat er niet meer dan 1x per mogelijkheid wordt opgeteld.
if number_of_vals_present_text > 1 or number_of_vals_present_datetime > 1:
    no_multiple_empty_vals = False

print(f"Test F: Er is maximaal 1 waarde voor informatie die niet is ingevuld: {no_multiple_empty_vals}")

Waarde ? is in gebruik in een tekstveld
Test F: Er is maximaal 1 waarde voor informatie die niet is ingevuld: True


## Dimensie 3: Waardenvaliditeit (inhoudelijke correctheid)
* Vraag: Zijn er kolommen waarvan de waarden volgens een bepaalde standaard zijn ingevuld? Zo ja, welke?
* Vraag: Zijn er kolommen die niet volgens een standaard zijn ingevuld, waarbij dat wel zou kunnen? Zo ja, welke?
    * Test A: Kolommen die een standaardwaardenlijst volgen, hebben alleen waarden die in deze standaardwaardenlijst voorkomen.
    * Test B: (tekst) Kolommen die volgens een gestandaardiseerde vorm worden ingevuld, hebben alleen waarden die aan die vorm voldoen
* Vraag: Komt de informatie die in een kolom staat overeen met wat de kolomnaam en/of data dictionary aangeeft?
* Vraag: Is de opbouw van tekst in de tekstuele kolommen consistent?
    * Test C: Er zijn geen kolommen waar geen data in staat.
    * Test D: Er zijn geen rijen die gedupliceerd zijn.
    * Test E: Er zijn geen kolommen waar in elke rij dezelfde data staat.
    * Test F: Er zijn geen (tekst) kolommen met waarden met whitespaces aan het begin of eind

In [219]:
test_resultaten_dim3 = []

In [220]:
# in de lijst staat: [0] welke dataset/schema naam, [1] welke lijst/tabel/ tabel naam, [2], welke eigenschap of kolom / kolom naam [3] uri naar dataset in het geval van linked data, [4] zou kunnen volgen (could), of volgt deze al (should)?
dict_col_with_standard_value = {
    "cpx_code_b": ["Archeologisch_Basis_Register", "complextypen", "Omschrijving", None, "should"],
    "cpx_code_o": ["Archeologisch_Basis_Register", "complextypen", "SIKB Code",None, "should"], # TODO welke is hier nou leidend? SIKB 0102 of ABR?
    "cpx_code_n": ["Archeologisch_Basis_Register", "complextypen", "SIKB Code", None, "should"],
    "cpx_begin" : ["Archeologisch_Basis_Register", "periode", "SIKB Code", None, "should"],
    "cpx_eind_p": ["Archeologisch_Basis_Register", "periode", "SIKB Code", None, "should"]
}
# in de lijst staat: [0] welke standaard (bijvoorbeeld ISO), [1] zou kunnen volgen (could) of volgt deze al (should)?
dict_col_with_standard_form = {
    "cpx_jaar": ["ISO 8601", "could"]
} # TODO: ik ken lang niet alle standaarden. Mogelijk moet dit worden uitgebreid

#### Vraag: Zijn er kolommen waarvan de waarden volgens een bepaalde standaard zijn ingevuld? Zo ja, welke? & Vraag: Zijn er kolommen die niet volgens een standaard zijn ingevuld, waarbij dat wel zou kunnen? Zo ja, welke?

In [221]:
print("gebruik van standaardwaardenlijsten:\n")
for item in dict_col_with_standard_value:
    if dict_col_with_standard_value[item][4] == "could":
        print(f"{item} zou gebruik kunnen maken van {dict_col_with_standard_value[item][0]} : {dict_col_with_standard_value[item][1]}")
    elif dict_col_with_standard_value[item][4] == "should":
        print(f"{item} maakt gebruik van {dict_col_with_standard_value[item][0]} : {dict_col_with_standard_value[item][1]}")
print("\ngebruik van gestandaardiseerde vormen: \n")
for item in dict_col_with_standard_form:
    if dict_col_with_standard_form[item][1] == "could":
        print(f"{item} zou gebruik kunnen maken van {dict_col_with_standard_form[item][0]}")
    elif dict_col_with_standard_form[item][1] == "should":
        print(f"{item} maakt gebruik van {dict_col_with_standard_form[item][0]}")


gebruik van standaardwaardenlijsten:

cpx_code_b maakt gebruik van Archeologisch_Basis_Register : complextypen
cpx_code_o maakt gebruik van Archeologisch_Basis_Register : complextypen
cpx_code_n maakt gebruik van Archeologisch_Basis_Register : complextypen
cpx_begin maakt gebruik van Archeologisch_Basis_Register : periode
cpx_eind_p maakt gebruik van Archeologisch_Basis_Register : periode

gebruik van gestandaardiseerde vormen: 

cpx_jaar zou gebruik kunnen maken van ISO 8601


#### Test A

In [222]:
# TODO Note T: hier heb ik nu gebruik gemaakt van de domeintabellen van de SIKB, maar dit zou gebruik moeten maken van de linked data van het ABR. Het was voor mij even te hoog gegrepen om ook SPARQL te leren om deze lijsten in te laden, dus dit zal iemand anders even moeten aanvullen.
# ABR versie

# activeer triple/sparql toegang in duckdb
# duckdb.sql(f"""INSTALL rdf FROM community;
#                 LOAD rdf;""") # NOTE: let op dit is een community extension

# maak een lijst van complextypen uit het ABR

# maak een lijst van perioden uit het ABR

# maak een lijst met ABRcodes voor complextypen uit het ABR

In [223]:
# SIKB versie # TODO vervangen met ABR linked data inladen.
# SIKB domeinlijst complextypen TODO zeer ad hoc! Geen voorbeeld van hoe het inladen van standaardwaardenlijsten in de eindversie/het DAP gedaan moet worden!
for item in dict_col_with_standard_value:
    if dict_col_with_standard_value[item][0] == "Archeologisch_Basis_Register":
        duckdb.sql(f"""CREATE SCHEMA IF NOT EXISTS {dict_col_with_standard_value[item][0]}""")
        if dict_col_with_standard_value[item][1] == "complextypen":
            duckdb.sql(f"""CREATE TABLE IF NOT EXISTS {dict_col_with_standard_value[item][0]}.{dict_col_with_standard_value[item][1]} AS SELECT * FROM read_csv('./data/SIKB_domeintabel_5_Complextype.csv') """) # file is lokaal geinstalleerd
        elif dict_col_with_standard_value[item][1] == "periode":
            duckdb.sql(f"""CREATE TABLE IF NOT EXISTS {dict_col_with_standard_value[item][0]}.{dict_col_with_standard_value[item][1]} AS SELECT * FROM read_csv('./data/SIKB_domeintabel_5_Periode.csv') """) # file is lokaal geinstalleerd


In [224]:
standard_lists_followed = True
for item in dict_col_with_standard_value:
    if dict_col_with_standard_value[item][4] == "should": #
        val = duckdb.sql(f"""
        SELECT
            LIST_HAS_ALL(
                (
                SELECT
                    LIST(DISTINCT({item}))
                FROM {table}
                )
                ,
                (
                SELECT
                    LIST("{dict_col_with_standard_value[item][2]}")
                FROM
                    {dict_col_with_standard_value[item][0]}.{dict_col_with_standard_value[item][1]}
                WHERE
                    "Eind geldigheid" IS NULL
                )
            );
        """).fetchone()[0]
        print(f"{item} volgt de beoogde standaardwaardenlijst: {val}")
        if not val:
            standard_lists_followed = False
print(f"Test A: Kolommen die een standaardwaardenlijst volgen, hebben alleen waarden die in deze standaardwaardenlijst voorkomen.: {standard_lists_followed}.\n"
      f"let op: alleen kolommen die al een standaardwaardenlijst volgen zijn getest. Kolommen die een standaardwaardenlijst ZOUDEN kunnen volgen worden niet getest.")
test_resultaten_dim3.append(standard_lists_followed)


cpx_code_b volgt de beoogde standaardwaardenlijst: False
cpx_code_o volgt de beoogde standaardwaardenlijst: False
cpx_code_n volgt de beoogde standaardwaardenlijst: False
cpx_begin volgt de beoogde standaardwaardenlijst: False
cpx_eind_p volgt de beoogde standaardwaardenlijst: False
Test A: Kolommen die een standaardwaardenlijst volgen, hebben alleen waarden die in deze standaardwaardenlijst voorkomen.: False.
let op: alleen kolommen die al een standaardwaardenlijst volgen zijn getest. Kolommen die een standaardwaardenlijst ZOUDEN kunnen volgen worden niet getest.


#### Test B

In [225]:
standard_forms_followed = True
for item in dict_col_with_standard_form:
    if dict_col_with_standard_form[item][1] == "should":
        if dict_col_with_standard_form[item][0]== "ISO 8601":
            # TODO nog veel meer standaarden toepassen
            regex = r'^\d{4}-\d{2}-\d{2}$|^(0[1-9]|1[0-2])\/(0[1-9]|[12]\d|3[01])\/(19|20)\d{2}$|^(0[1-9]|[12]\d|3[01])\/(0[1-9]|1[0-2])\/(19|20)\d{2}$|^(0[1-9]|[12]\d|3[01])\.(0[1-9]|1[0-2])\.(19|20)\d{2}$' # TODO my regex should probably be checked by someone who knows what they're doing...
            string_follows_standard = duckdb.sql(f"""
                SELECT
                    (
                    SELECT
                        COUNT()
                    FROM
                        {table}
                    WHERE
                        regexp_full_match({item}, '{regex}', 'i')
                    )
                    ==
                    (SELECT
                        COUNT()
                    FROM
                        {table}
                    );
                """).fetchone()[0]
            print(f"kolom {item} volgt ISO 8601: {string_follows_standard}")
            if not string_follows_standard:
                standard_forms_followed = False
            # duckdb volgt in hun date time datatypen al automatisch ISO 8601, dus dit hoeft niet getest te worden. Alleen als de datums nog in een tekstkolom staan.
print(f"Test B: (tekst) Kolommen die volgens een gestandaardiseerde vorm worden ingevuld, hebben alleen waarden die aan die vorm voldoen: {standard_forms_followed}.\n"
      f"let op: alleen kolommen die al een standaardvorm volgen zijn getest. Kolommen die een standaardvorm ZOUDEN kunnen volgen worden niet getest.")
test_resultaten_dim3.append(standard_forms_followed)


Test B: (tekst) Kolommen die volgens een gestandaardiseerde vorm worden ingevuld, hebben alleen waarden die aan die vorm voldoen: True.
let op: alleen kolommen die al een standaardvorm volgen zijn getest. Kolommen die een standaardvorm ZOUDEN kunnen volgen worden niet getest.


#### Vraag: Komt de informatie die in een kolom staat overeen met wat de kolomnaam en/of data dictionary aangeeft?
Ja, al is van cpx_monito en cpx_jaar niet te achterhalen welke informatie hier precies zou moeten staan.

In [226]:
# hier kan verder qua code niks mee te worden gedaan, dit is puur interpretatie.
duckdb.sql(f"""SELECT * FROM {table} LIMIT 10 """).fetchdf()

,complex_id,terreinnum,rijksmonum,cma,cma_volgnr,cpx_monito,cpx_jaar,cpx_code_b,cpx_code_o,cpx_code_n,...,cpx_beschr,cpx_kwets,cpx_restau,cpx_consol,cpx_diepte,cpx_diep00,cpx_x_coor,cpx_y_coor,cpx_opmerk,geometry
0,52.0,60.0,330191.0,67A-002,None,3324,None,Klooster,RKLO,CTHD.KLO,...,Resten van een klooster,onbekend,None,None,0.0,0.0,19494.10005,366239.100025,None,"[1, 235, 3, 0, 0, 1, 0, 0, 0, 13, 0, 0, 0, 150..."
1,54.0,61.0,330202.0,67A3,None,3327,None,begraving,GX,BGV,...,Sporen van begraving,onbekend,None,None,0.0,0.0,19789.50010,366276.499975,None,"[1, 238, 3, 0, 0, 2, 0, 0, 0, 1, 235, 3, 0, 0,..."
2,53.0,61.0,330202.0,67A3,None,3326,None,Bewoning (inclusief verdediging),NX,BEWV,...,Sporen van bewoning,onbekend,None,None,0.0,0.0,19789.50010,366276.499975,None,"[1, 238, 3, 0, 0, 2, 0, 0, 0, 1, 235, 3, 0, 0,..."
3,56.0,62.0,330373.0,67A-004,None,3332,None,Weg,IWEG,INFR.WEG,...,Resten van een weg,onbekend,None,None,0.0,0.0,19883.89990,366405.600100,None,"[1, 235, 3, 0, 0, 1, 0, 0, 0, 7, 0, 0, 0, 94, ..."
4,57.0,62.0,330373.0,67A-004,None,3331,None,Bewoning (inclusief verdediging),NX,BEWV,...,Sporen van bewoning,onbekend,None,None,0.0,0.0,19883.89990,366405.600100,None,"[1, 235, 3, 0, 0, 1, 0, 0, 0, 7, 0, 0, 0, 94, ..."
5,55.0,62.0,330373.0,67A-004,None,3330,None,Bewoning (inclusief verdediging),NX,BEWV,...,Sporen van bewoning,onbekend,None,None,0.0,0.0,19883.89990,366405.600100,None,"[1, 235, 3, 0, 0, 1, 0, 0, 0, 7, 0, 0, 0, 94, ..."
6,59.0,63.0,330379.0,67A-005,None,3335,None,Legerplaats,VLP,BEWV.VLP,...,Resten van een legerplaats,onbekend,None,None,0.0,0.0,19691.99995,366465.900025,None,"[1, 235, 3, 0, 0, 1, 0, 0, 0, 42, 0, 0, 0, 127..."
7,60.0,63.0,330379.0,67A-005,None,3341,None,Weg,IWEG,INFR.WEG,...,Resten van een weg,onbekend,None,None,0.0,0.0,19691.99995,366465.900025,None,"[1, 235, 3, 0, 0, 1, 0, 0, 0, 42, 0, 0, 0, 127..."
8,61.0,63.0,330379.0,67A-005,None,3340,None,Wal/omwalling,VWAL,Wal,...,Resten van een wal,onbekend,None,None,0.0,0.0,19691.99995,366465.900025,None,"[1, 235, 3, 0, 0, 1, 0, 0, 0, 42, 0, 0, 0, 127..."
9,58.0,63.0,330379.0,67A-005,None,3333,None,Bewoning (inclusief verdediging),NX,BEWV,...,Sporen van bewoning,onbekend,None,None,0.0,0.0,19691.99995,366465.900025,None,"[1, 235, 3, 0, 0, 1, 0, 0, 0, 42, 0, 0, 0, 127..."


#### Vraag: Is de opbouw van tekst in de tekstuele kolommen consistent?

In [227]:
# geen testbare vraag zonder eerst handmatig de patronen te identificeren, dus niet te automatiseren. Voor nu overgeslagen.

#### Test C

In [228]:
no_null_columns = True
list_of_cols = duckdb.sql(f"""
                          SELECT column_name FROM (DESCRIBE {table})
                          """).fetchall()
for col in list_of_cols:
    val = duckdb.sql(f"""
    SELECT
        (
        SELECT
            COUNT()
        FROM
            {table}
        WHERE
            {col[0]} IS NULL
        )
        !=
        (
        SELECT
            COUNT()
        FROM
            {table}
        );""").fetchone()[0]
    if not val:
        no_null_columns = False
        print(f"kolom {col[0]} bestaat uit alleen maar NULL waarden")

print(f" Test C: Er zijn geen kolommen waar geen data in staat: {no_null_columns}")
test_resultaten_dim3.append(no_null_columns)


 Test C: Er zijn geen kolommen waar geen data in staat: True


#### Test D

In [229]:
list_of_cols = duckdb.sql(f"""
                          SELECT column_name FROM (DESCRIBE {table})
                          """).fetchdf()["column_name"].to_list()

no_duplicate_rows = duckdb.sql(f"""
SELECT
    (SELECT COUNT(DISTINCT({', '.join(list_of_cols)})) FROM {table})
    ==
    (SELECT COUNT() FROM {table});
    """).fetchone()[0]


print(f"Test D: Er zijn geen rijen die volledig gedupliceerd zijn: {no_duplicate_rows}")
test_resultaten_dim3.append(no_duplicate_rows)


Test D: Er zijn geen rijen die volledig gedupliceerd zijn: True


#### Test E

In [230]:
no_single_val_cols = True
list_of_cols = duckdb.sql(f"""
                          SELECT column_name FROM (DESCRIBE {table})
                          """).fetchall()
for col in list_of_cols:
    val = duckdb.sql(f"""
    SELECT
     (SELECT COUNT(DISTINCT({col[0]})) FROM {table})
     >1""")
    if not val:
        no_single_val_cols = False


print(f"Test E: Er zijn geen kolommen waar in elke rij dezelfde data staat: {no_single_val_cols}")
test_resultaten_dim3.append(no_single_val_cols)


Test E: Er zijn geen kolommen waar in elke rij dezelfde data staat: True


#### Test F

In [231]:
no_trailing_or_leading_whitespaces = True
for item in dict_datatypes:
    if item == 'string':
        for type in dict_datatypes[item]:
            list_of_cols = duckdb.sql(f"""
                                            SELECT column_name FROM (DESCRIBE {table}) WHERE column_type == '{type}'
                                            """).fetchall()
            for item in list_of_cols:
                val = duckdb.sql(f"""
                SELECT
                    CASE WHEN EXISTS(
                        SELECT
                            1
                        FROM
                            {table}
                        WHERE
                            ends_with({item[0]}, ' ')
                            OR
                            starts_with({item[0]}, ' ')
                        )
                    THEN False
                    ELSE True
                    END;
                    """).fetchone()[0]
                if not val:
                    no_trailing_or_leading_whitespaces = False


print(f"Test F: Er zijn geen (tekst) kolommen met waarden met whitespaces aan het begin of eind: {no_trailing_or_leading_whitespaces}")
test_resultaten_dim3.append(no_trailing_or_leading_whitespaces)


Test F: Er zijn geen (tekst) kolommen met waarden met whitespaces aan het begin of eind: True


#### Resultaten dimensie 3

In [232]:
aantal_testen_succes_dim3 = 0

for test in test_resultaten_dim3:
    if test:
        aantal_testen_succes_dim3 += 1

if len(test_resultaten_dim3) == aantal_testen_succes_dim3:
    print(f"Er zijn {len(test_resultaten_dim3)} tests uitgevoerd voor dimensie 3. Daarvan is/zijn aan {aantal_testen_succes_dim3} test(en) voldaan. Dat betekent dat aan datakwaliteitsdimensie 3 in zijn geheel is voldaan")
    dk_dimensies_voldaan +=1
else:
    print(f"Er zijn {len(test_resultaten_dim3)} tests uitgevoerd voor dimensie 3. Daarvan is/zijn aan {aantal_testen_succes_dim3} test(en) voldaan. Dat betekent dat aan datakwaliteitsdimensie 3 NIET in zijn geheel is voldaan.")



Er zijn 6 tests uitgevoerd voor dimensie 1. Daarvan is/zijn aan 5 test(en) voldaan. Dat betekent dat aan datakwaliteitsdimensie 1 NIET in zijn geheel is voldaan.


## Dimensie 4: Granulariteit en eenduidige representatie
* Vraag: Is het niveau van detail van de data _binnen_ kolommen hetzelfde?
    * Test A: Is de eenheid van de kolom consistent? (Notitie Thya: Dit kan alleen getest worden als de eenheid ook in die kolom of een andere kolom is vastgelegd).
* Zijn er geaggregeerde kolommen (samentrekkingen van 2 kolommen die beiden in de dataset staan)
    * Test B: Er zijn geen kolommen die bestaan uit een samentrekking van twee andere kolommen
    * Test C: Er zijn geen verborgen aggregaties (zelfde als test
* Zijn er kolommen waarin andere kolommen met elkaar verrekend zijn? (en zo ja, is dat dan ongewenst?) notitie Thya: Er kunnen hier testen uitgevoerd worden, maar er zijn zo veel verschillende manieren van verrekenen, dat dit nooit volledig gecheckt zou kunnen worden. Bovendien is het niet altijd slecht, dus dit is echt op oordeel van de domeindeskundige)
* Zijn er kolommen waarbij het niveau van detail niet voldoet voor het beoogde doel?
* Staan de waarden en de indicatie van precisie van deze waarde in losse kolommen? Zo ja, is het duidelijk welke precisiekolom bij welke kolom hoort (ook wanneer kolommen gehusseld worden)?
* Is van kolommen waar topografische benamingen in staat duidelijk over welk type geografische eenheid dit gaat?
    * Zo ja, staan er in die kolom inderdaad alleen dat type topografische benaming?
        * Test B: Staat in een kolom met provincies alleen provincies? (inclusief historische)
        * Test C: Staat in een kolom met gemeenten alleen gemeenten? (inclusief historische)
        * Test D: Staat in een kolom met plaatsen alleen plaatsen? (inclusief historische)
        * Test E: Staat in een kolom met landen alleen landen? (inclusief historische en niet-erkende)

## Dimensie 5: Semantische eenduidigheid en interoperabiliteit
* Is er een data dictionary?
* Is de data dictionary up-to-date?
* Is er vastgelegd of de kolommen volgens een (standaard) waardenlijst moet worden ingevuld?
* Zijn de referentielijsten expliciet gekoppeld aan de informatie?
* Is voor de waardenlijsten die via linked data (o.i.d.) worden binnengehaald een kolom met uri's aanwezig?
* Zijn er kolommen waar zowel standaard waarden als vrije velden is staan?


## Dimensie 6: Ruimtelijke referentie-eenduidigheid

Alleen van toepassing als het een ruimtelijke dataset betreft (maar let op: ook niet GIS-ready datasets bevatten vaak ruimtelijke informatie)
* Zijn locaties expliciet gemaakt, of zijn er alleen omschrijvingen.
* Is de ruimtelijke informatie binnen kolommen op hetzelfde niveau van detail (als het geen geometrie betreft)
* Is het coordinaatreferentiestelsel gedefinieerd?
*

## Dimensie 7: Ruimtelijke schaal en resolutie consistentie
* Is het schaalniveau van de geometrie gedefinieerd (voor vector datasets)
* Is de resolutie gedefinieerd (voor raster bestanden)
* Heeft de geometrie van de dataset over het gehele bestand hetzelfde schaalniveau/resolutie?
* Past het schaalniveau/de resolutie bij het beoogde doel?
* Komt de precisie van de geometrie overeen met het schaalniveau?

## Dimensie 8: Ruimtelijke geometrische correctheid
* Is het geometrietype expliciet?
* Zijn de geometrieën valide (niet zelfkruisend/niet niet aansluitend/geen gaten)
* Hebben alle geometrieën ook geassocieerde data?

## Dimensie 9: Temporele resolutie
* Zijn er kolommen voor geldigheid?
* Zijn er kolommen voor wanneer een record is toegevoegd?
* Worden "oude" rijen bewaard?
* Gebruiken alle datum kolommen dezelfde vorm? (en zo nee, is dit een probleem?)
* Is eind_geldigheid altijd na begin_geldigheid?
* Zijn eind_geldigheid en begin_geldigheid altijd na registratiedatum?
* Zijn er verschillende kolommen voor verschillende temporele detailniveaus?